In [2]:
import numpy as np
import pandas as pd
from scipy.interpolate import pchip_interpolate
from scipy.optimize import minimize, Bounds, LinearConstraint
from types import SimpleNamespace

def processTrack(filename):
    """
    processTrack - 5-Line Surface Version (Fixed Output Compatibility)
    Calculates the Minimum Curvature Path (MCP) and track geometry.
    """
    
    # 1. Import Data
    # Assuming CSV has headers, so we skip the first row (header=0 in Pandas)
    raw = pd.read_csv(filename).values
    name = 'Aragon'
    
    # --- COLUMN MAPPING (0-indexed in Python) ---
    x_L_out = raw[:, 1];  y_L_out = raw[:, 2];  z_L_out = raw[:, 3]
    x_L_in  = raw[:, 4];  y_L_in  = raw[:, 5];  z_L_in  = raw[:, 6]
    x_C     = raw[:, 7];  y_C     = raw[:, 8];  z_C     = raw[:, 9]
    x_R_in  = raw[:, 10]; y_R_in  = raw[:, 11]; z_R_in  = raw[:, 12]
    x_R_out = raw[:, 13]; y_R_out = raw[:, 14]; z_R_out = raw[:, 15]
    
    # 1.2 Force Loop Closure
    gap_dist = np.hypot(x_C[0] - x_C[-1], y_C[0] - y_C[-1])
    if gap_dist > 0.1:
        # Append first point to the end to close the loop
        x_L_out = np.append(x_L_out, x_L_out[0]); y_L_out = np.append(y_L_out, y_L_out[0]); z_L_out = np.append(z_L_out, z_L_out[0])
        x_L_in  = np.append(x_L_in, x_L_in[0]);   y_L_in  = np.append(y_L_in, y_L_in[0]);   z_L_in  = np.append(z_L_in, z_L_in[0])
        x_C     = np.append(x_C, x_C[0]);         y_C     = np.append(y_C, y_C[0]);         z_C     = np.append(z_C, z_C[0])
        x_R_in  = np.append(x_R_in, x_R_in[0]);   y_R_in  = np.append(y_R_in, y_R_in[0]);   z_R_in  = np.append(z_R_in, z_R_in[0])
        x_R_out = np.append(x_R_out, x_R_out[0]); y_R_out = np.append(y_R_out, y_R_out[0]); z_R_out = np.append(z_R_out, z_R_out[0])

    # 2. Pre-processing & Interpolation
    centerXY = np.column_stack((x_C, y_C))
    stepLengths = np.sqrt(np.sum(np.diff(centerXY, axis=0)**2, axis=1))
    stepLengths = np.insert(stepLengths, 0, 0.0) # Prepend 0
    cumulativeLen = np.cumsum(stepLengths)
    
    nseg = 1500
    finalStepLocs = np.linspace(0, cumulativeLen[-1], nseg)
    
    # Helper to mimic MATLAB's interp1(..., 'pchip')
    def interp_track(v):
        return pchip_interpolate(cumulativeLen, v, finalStepLocs)
        
    xl_out = interp_track(x_L_out);  yl_out = interp_track(y_L_out);  zl_out = interp_track(z_L_out)
    xl_in  = interp_track(x_L_in);   yl_in  = interp_track(y_L_in);   zl_in  = interp_track(z_L_in)
    xc     = interp_track(x_C);      yc     = interp_track(y_C);      zc     = interp_track(z_C)
    xr_in  = interp_track(x_R_in);   yr_in  = interp_track(y_R_in);   zr_in  = interp_track(z_R_in)
    xr_out = interp_track(x_R_out);  yr_out = interp_track(y_R_out);  zr_out = interp_track(z_R_out)

    # 3. Solver Setup (3D "Best Fit Plane")
    xin = xr_out
    yin = yr_out
    xout = xl_out
    yout = yl_out
    
    delx = xout - xin
    dely = yout - yin
    
    n = len(xin)
    zin_fit = np.zeros(n)
    delz_fit = np.zeros(n)
    
    for i in range(n):
        total_w = np.hypot(xl_out[i] - xr_out[i], yl_out[i] - yr_out[i])
        if total_w < 1e-6:
            total_w = 1.0
            
        d1 = 0
        d2 = np.hypot(xr_in[i] - xr_out[i], yr_in[i] - yr_out[i]) / total_w
        d3 = np.hypot(xc[i] - xr_out[i], yc[i] - yr_out[i]) / total_w
        d4 = np.hypot(xl_in[i] - xr_out[i], yl_in[i] - yr_out[i]) / total_w
        d5 = 1.0
        
        a_vals = [d1, d2, d3, d4, d5]
        z_slice = [zr_out[i], zr_in[i], zc[i], zl_in[i], zl_out[i]]
        
        p = np.polyfit(a_vals, z_slice, 1)
        delz_fit[i] = p[0]
        zin_fit[i] = p[1]
        
    delz = delz_fit
    zin = zin_fit
    
    # Matrix Definition & MCP Solver
    H = np.zeros((n, n))
    B = np.zeros(n)
    
    for i in range(1, n-1):
        # i-1 Interaction
        H[i-1, i-1] += delx[i-1]**2 + dely[i-1]**2 + delz[i-1]**2
        H[i-1, i]   -= 2*delx[i-1]*delx[i] + 2*dely[i-1]*dely[i] + 2*delz[i-1]*delz[i]
        H[i-1, i+1] += delx[i-1]*delx[i+1] + dely[i-1]*dely[i+1] + delz[i-1]*delz[i+1]
        
        # i Interaction
        H[i, i-1]   -= 2*delx[i-1]*delx[i] + 2*dely[i-1]*dely[i] + 2*delz[i-1]*delz[i]
        H[i, i]     += 4*delx[i]**2 + 4*dely[i]**2 + 4*delz[i]**2
        H[i, i+1]   -= 2*delx[i]*delx[i+1] + 2*dely[i]*dely[i+1] + 2*delz[i]*delz[i+1]
        
        # i+1 Interaction
        H[i+1, i-1] += delx[i-1]*delx[i+1] + dely[i-1]*dely[i+1] + delz[i-1]*delz[i+1]
        H[i+1, i]   -= 2*delx[i]*delx[i+1] + 2*dely[i]*dely[i+1] + 2*delz[i]*delz[i+1]
        H[i+1, i+1] += delx[i+1]**2 + dely[i+1]**2 + delz[i+1]**2
        
        termX = (xin[i+1] + xin[i-1] - 2*xin[i])
        termY = (yin[i+1] + yin[i-1] - 2*yin[i])
        termZ = (zin[i+1] + zin[i-1] - 2*zin[i])
        
        B[i-1] += 2*termX*delx[i-1] + 2*termY*dely[i-1] + 2*termZ*delz[i-1]
        B[i]   -= 4*termX*delx[i]   + 4*termY*dely[i]   + 4*termZ*delz[i]
        B[i+1] += 2*termX*delx[i+1] + 2*termY*dely[i+1] + 2*termZ*delz[i+1]

    # --- Python Optimization (Replaces quadprog) ---
    # We want to minimize: x^T * H * x + B^T * x
    def objective(x):
        return x.T @ H @ x + B @ x
        
    def derivative(x):
        return 2 * H @ x + B

    bounds = Bounds(np.zeros(n), np.ones(n))
    
    # Aeq constraint: x[0] - x[-1] = 0 (Start and end must match)
    Aeq = np.zeros((1, n))
    Aeq[0, 0] = 1
    Aeq[0, -1] = -1
    eq_cons = LinearConstraint(Aeq, [0], [0])
    
    x0 = np.full(n, 0.5) # Initial guess in the middle of the track
    
    print("Solving Minimum Curvature Path (this may take a moment)...")
    res = minimize(objective, x0, method='SLSQP', jac=derivative, bounds=bounds, constraints=[eq_cons], options={'disp': False})
    resMCP = res.x
    
    x_opt = xin + resMCP * delx
    y_opt = yin + resMCP * dely
    z_opt = zin + resMCP * delz
    
    # 4. Advanced Z & Banking Calculation
    banking_angle = np.zeros(n)
    for i in range(n):
        width_i = np.hypot(xout[i] - xin[i], yout[i] - yin[i])
        z_ratio = np.clip(delz[i] / width_i if width_i > 1e-6 else 0, -1, 1)
        banking_angle[i] = np.arctan(z_ratio)
        
    # 5. Corner Radius Solver
    RProfileLap = np.zeros(n)
    TSignLap = np.zeros(n)
    
    for i in range(n):
        u1 = n - 2 if i == 0 else i - 1
        u2 = i
        u3 = 1 if i == n - 1 else i + 1
            
        x1, y1 = x_opt[u1], y_opt[u1]
        x2, y2 = x_opt[u2], y_opt[u2]
        x3, y3 = x_opt[u3], y_opt[u3]
        
        a_len = np.hypot(x2 - x3, y2 - y3)
        b_len = np.hypot(x1 - x3, y1 - y3)
        c_len = np.hypot(x1 - x2, y1 - y2)
        
        area2 = (x1*(y2 - y3) + x2*(y3 - y1) + x3*(y1 - y2))
        A = 0.5 * abs(area2)
        
        if A > 0:
            R_mag = (a_len * b_len * c_len) / (4 * A)
            turnSign = np.sign(area2)
        else:
            R_mag = np.inf
            turnSign = 0
            
        RProfileLap[i] = R_mag
        TSignLap[i] = turnSign

    # 6. Drive Cycle Generator Path
    nLaps = 1
    xresMCP_laps = np.tile(x_opt, nLaps)
    yresMCP_laps = np.tile(y_opt, nLaps)
    TSignProfile = np.tile(TSignLap, nLaps)
    
    dx_seg = np.diff(xresMCP_laps)
    dy_seg = np.diff(yresMCP_laps)
    segmentLengths = np.sqrt(dx_seg**2 + dy_seg**2)
    lapLength = np.sum(segmentLengths)
    
    targetLap = 5078 * nLaps
    scale_factor = targetLap / lapLength
    
    xresMCP_laps *= scale_factor
    yresMCP_laps *= scale_factor
    xin *= scale_factor; yin *= scale_factor
    xout *= scale_factor; yout *= scale_factor
    xc *= scale_factor; yc *= scale_factor
    x_opt *= scale_factor; y_opt *= scale_factor; z_opt *= scale_factor
    
    RProfile = np.tile(RProfileLap, nLaps) * scale_factor
    dx_seg = np.diff(xresMCP_laps)
    dy_seg = np.diff(yresMCP_laps)
    segmentLengths = np.sqrt(dx_seg**2 + dy_seg**2)
    
    # 7. Output Packaging (Using SimpleNamespace to mimic MATLAB structs)
    trackDataOut = SimpleNamespace(
        xresMCP_laps=xresMCP_laps,
        yresMCP_laps=yresMCP_laps,
        RProfile=RProfile,
        TSignProfile=TSignProfile,
        segmentLengths=segmentLengths,
        zt=z_opt,
        xt=xc,
        yt=yc,
        xin=xin, xout=xout,
        yin=yin, yout=yout,
        xresMCP=x_opt,
        yresMCP=y_opt,
        finalStepLocs=finalStepLocs,
        scale_factor=scale_factor,
        name=name,
        banking=banking_angle
    )
    
    return trackDataOut

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import pchip_interpolate, interp1d
from scipy.ndimage import gaussian_filter1d

# NOTE: You will need to bring over or translate your 'processTrack' logic.
# For now, we assume it's imported from another Python file.
# from track_processor import processTrack

def bzero_v4_release(gear_ratios=None, is_direct_drive=True, sprockets=(14, 45), is_debug=True, track_filename='high_fidel_track.csv'):
    """
    Calculates lap time, energy consumption and top speed for a motorcycle.
    """
    if gear_ratios is None:
        gear_ratios = [2.5, 1.8, 1.2, 0.77]
    
    debug_mode = is_debug

    # =========================================================================
    # 1.1 Basic Cleanup/Init
    # =========================================================================
    # trackDataOut = processTrack(track_filename) # Uncomment when processTrack is translated
    
    # --- MOCK DATA FOR COMPILATION (Replace with your actual processTrack output) ---
    class MockTrack: pass
    track = MockTrack()
    num_pts = 1000
    track.xresMCP_laps = np.linspace(0, 1000, num_pts)
    track.yresMCP_laps = np.linspace(0, 1000, num_pts)
    track.segmentLengths = np.ones(num_pts - 1)
    track.RProfile = np.full(num_pts, np.inf)
    track.TSignProfile = np.ones(num_pts)
    track.zt = np.zeros(num_pts)
    track.banking = np.zeros(num_pts)
    track.xin, track.xout, track.yin, track.yout = [np.zeros(num_pts) for _ in range(4)]
    track.xt, track.yt = np.zeros(num_pts), np.zeros(num_pts)
    track.xresMCP, track.yresMCP = np.zeros(num_pts), np.zeros(num_pts)
    track.finalStepLocs = np.arange(num_pts)
    track.scale_factor = 1.0
    # -------------------------------------------------------------------------

    # Extract track variables
    xresMCP_laps = track.xresMCP_laps
    yresMCP_laps = track.yresMCP_laps
    segmentLengths = track.segmentLengths
    RProfile = track.RProfile.copy()
    TSignProfile = track.TSignProfile
    zt = track.zt
    bankingNoise = track.banking

    # Link start/finish (Infinite straights)
    RProfile[:20] = np.inf
    RProfile[-20:] = np.inf
    RProfile[RProfile < 0.1] = np.inf
    
    raw_curvature = 1.0 / RProfile
    raw_curvature[np.isnan(raw_curvature)] = 0

    # Smooth curvature (MATLAB smoothdata gaussian -> SciPy gaussian_filter1d)
    # MATLAB window of 10 is roughly a sigma of 2
    clean_curvature = gaussian_filter1d(raw_curvature, sigma=2)
    
    # Convert back to radius
    # Ignore divide by zero warnings temporarily for straight lines
    with np.errstate(divide='ignore'):
        RProfile_Clean = 1.0 / clean_curvature
    
    max_straight_R = 100000000
    RProfile_Clean[np.abs(RProfile_Clean) > max_straight_R] = max_straight_R

    s_track = np.zeros(len(zt))
    for idx in range(len(segmentLengths)):
        s_track[idx+1] = s_track[idx] + segmentLengths[idx]

    # Smoothing banking profile
    distance_vec = s_track[:-1]
    avg_dx = np.mean(np.diff(distance_vec)) if len(distance_vec) > 1 else 1.0
    window_distance_m = 45
    window_points = max(1, int(round(window_distance_m / avg_dx)))
    bankingProfile = gaussian_filter1d(bankingNoise, sigma=window_points/5)

    # 1st and 2nd derivatives
    dz_ds = np.gradient(zt, s_track)
    d2z_ds2 = np.gradient(dz_ds, s_track)
    
    # Vertical curvature K
    K_vert = d2z_ds2 / (1 + dz_ds**2)**1.5

    # =========================================================================
    # 1.2 Specific Constants
    # =========================================================================
    wheelRadius = 0.601 / 2
    frontalArea = 0.3
    P_aux = 150
    carMass = 220
    Me_scalingfactor = 1.1
    M_effective = carMass * Me_scalingfactor
    speed_limit = 85
    maxV = 85
    cd = 0.4
    rho = 1.225
    h_cog = 0.45
    maxLeanRad = np.deg2rad(55)
    maxLeanRate = np.deg2rad(30)
    g = 9.81
    Ad = 0.004
    Bd = 0.000025
    useLeanRateClamp = True
    useMaxLeanClamp = True
    
    finalDriveRatio = 3.68
    chainDriveRatio = sprockets[1] / sprockets[0]
    
    if is_direct_drive or np.isscalar(gear_ratios) or len(gear_ratios) == 1:
        mech_eff = 0.96
        is_direct_drive = True
    else:
        mech_eff = 0.93

    shift_delay_time = 0.050
    shift_timer = 0
    current_gear = 0  # 0-indexed in Python (was 1 in MATLAB)
    shift_cooldown_time = 1.0
    shift_cooldown_timer = 0

    rpm_table = np.array([0, 250, 500, 750, 1000, 1250, 1500, 1750, 2000, 2250, 2500, 2750, 3000, 3250, 3500, 3750, 4000, 4250, 4500, 4750, 5000, 5250, 5500, 5750, 6000, 6250, 6500, 6750, 7000, 7250, 7500])
    T_max_table = np.array([120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 120, 114.592, 107.851, 101.859, 96.498, 91.673, 87.308, 83.339, 79.716, 76.394, 73.339, 70.518, 67.906, 65.481, 63.223, 61.115])
    eta_max_table = np.array([0, 0.9082, 0.9112, 0.914, 0.9166, 0.9189, 0.921, 0.9229, 0.9246, 0.926, 0.9272, 0.9282, 0.929, 0.9296, 0.9299, 0.93, 0.9299, 0.9296, 0.929, 0.9282, 0.9272, 0.926, 0.9246, 0.9229, 0.921, 0.9189, 0.9166, 0.914, 0.9112, 0.9082, 0.905])
    motor_redline = 7500
    
    t_tyre = 67.8 / 1000
    tireFrictionCoeff = 1.4
    leanDeg_AreaTable = np.array([0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55])
    contactAreaTable = np.array([0.0054, 0.0054, 0.0054, 0.0055, 0.0055, 0.0056, 0.0057, 0.0058, 0.0059, 0.0061, 0.0063, 0.0065])
    
    rotor_OD = 0.320
    rotor_ID = 0.246
    R_eff = 0.5 * (rotor_OD + rotor_ID)
    D_piston = 33.9e-3
    N_piston = 4
    A_piston = N_piston * np.pi * (D_piston / 2)**2
    mu_pad = 0.4
    lineP_max = 1e6
    T_brake_max = 2 * mu_pad * lineP_max * A_piston * R_eff
    F_brake_wheel_max = T_brake_max / wheelRadius

    # =========================================================================
    # 1.3 Init starting conditions
    # =========================================================================
    velocity = 0.1
    time_val = 0
    totalEnergy_J = 0
    
    numSteps = len(xresMCP_laps) - 1
    
    # Preallocation
    FCMDprofile = np.zeros(numSteps)
    DTprofile = np.zeros(numSteps)
    velocityProfile = np.zeros(numSteps)
    timeProfile = np.zeros(numSteps)
    accel = np.zeros(numSteps)
    F_power_array = np.zeros(numSteps)
    F_remain_array = np.zeros(numSteps)
    F_applied = np.zeros(numSteps)
    FTyreprofile = np.zeros(numSteps)
    Aprofile = np.zeros(numSteps)
    Muprofile = np.zeros(numSteps)
    powerProfile = np.zeros(numSteps)
    leanProfile = np.zeros(numSteps)

    # =========================================================================
    # 2. Best possible velocity profile VLIM
    # =========================================================================
    leanRad_AreaTable = np.deg2rad(leanDeg_AreaTable)
    A0 = contactAreaTable[0]
    mu_available_curve = tireFrictionCoeff * (contactAreaTable / A0)
    mu_required_curve = np.tan(leanRad_AreaTable)
    diff_curve = mu_available_curve - mu_required_curve
    
    interp_angles = np.linspace(0, np.deg2rad(55), 1000)
    diff_interp = pchip_interpolate(leanRad_AreaTable, diff_curve, interp_angles)
    idx_cross = np.argmin(np.abs(diff_interp))
    limit_lean_angle = interp_angles[idx_cross]
    mu_at_limit = pchip_interpolate(leanRad_AreaTable, mu_available_curve, limit_lean_angle)
    mu_corn_dynamic = mu_at_limit * 0.98
    
    VLIMprofile = np.zeros(numSteps)
    a_mech_limit = F_brake_wheel_max / M_effective
    safety_factor = 1.0
    phi_prev = 0

    # 2.1 Geometric limit
    for k in range(numSteps):
        R_k = np.abs(RProfile_Clean[k])
        if np.isfinite(R_k) and R_k > 0 and R_k < 10000:
            turnSign = TSignProfile[k]
            theta_bank = bankingProfile[k]
            
            theta_bank_effective = np.abs(theta_bank) if np.sign(theta_bank) == np.sign(turnSign) else -np.abs(theta_bank)
            
            num = mu_corn_dynamic + np.tan(theta_bank_effective)
            den = max(0.01, 1 - mu_corn_dynamic * np.tan(theta_bank_effective))
            v_friction = np.sqrt(g * R_k * (num / den))
            
            real_max_lean_rad = min(maxLeanRad, limit_lean_angle)
            effective_max_lean = real_max_lean_rad + theta_bank_effective
            if effective_max_lean >= np.pi/2: effective_max_lean = np.pi/2 - 0.01
            v_lean_geometry = np.sqrt(g * R_k * np.tan(effective_max_lean))
            
            VLIMprofile[k] = min(v_friction, v_lean_geometry, speed_limit)
        else:
            VLIMprofile[k] = speed_limit

    # 2.2 Backwards Pass
    for pass_idx in range(3):
        for k in range(len(VLIMprofile)-2, -1, -1): # Backwards iteration
            dist = segmentLengths[k]
            v_next = VLIMprofile[k+1]
            R_here = np.abs(RProfile_Clean[k])
            turnSign = TSignProfile[k]
            theta_bank_effective = np.sign(turnSign) * bankingProfile[k]
            
            v_est_mid = np.sqrt(v_next**2 + a_mech_limit * dist)
            
            if np.isfinite(R_here) and R_here > 0 and R_here < 10000:
                phi_flat = np.arctan(v_est_mid**2 / (g * R_here))
                phi_ideal = phi_flat - theta_bank_effective
                if (h_cog > t_tyre) and (phi_ideal != 0):
                    phi_delta = np.arcsin((t_tyre * np.sin(phi_ideal)) / (h_cog - t_tyre))
                    phi = phi_ideal + phi_delta
                else:
                    phi = phi_ideal
            else:
                phi = 0
                
            phi_relative_to_road = np.rad2deg(np.abs(phi))
            
            # Linear extrap handler
            interp_func = interp1d(leanDeg_AreaTable, contactAreaTable, fill_value="extrapolate")
            A_contact = interp_func(phi_relative_to_road)
            
            Aprofile[k] = A_contact
            mu_eff = tireFrictionCoeff * (A_contact / A0)**0.15
            
            dz = zt[k+1] - zt[k]
            sin_theta = dz / np.sqrt(dist**2 + dz**2)
            cos_theta = np.sqrt(1 - sin_theta**2)
            
            K_v = K_vert[k]
            Fz_vert = carMass * (v_est_mid**2) * K_v
            
            Fz_bp = (carMass * g * np.cos(theta_bank_effective) * cos_theta) + \
                    (carMass * (v_est_mid**2 / max(1.0, R_here)) * np.sin(theta_bank_effective)) + Fz_vert
            Fz_bp = max(0.1, Fz_bp)
            
            f_friction = mu_eff * Fz_bp
            a_grip_max = f_friction / M_effective
            v_est = np.sqrt(v_next**2 + 2 * a_mech_limit * dist)
            v_mean_bp = (v_next + v_est) / 2
            
            a_lat_geometric = v_mean_bp**2 / max(1.0, R_here)
            a_banking_assist = g * np.tan(theta_bank_effective)
            a_lat_demand_tire = np.abs(a_lat_geometric - a_banking_assist)
            f_lat_demand = carMass * a_lat_demand_tire
            a_lat_scaled = f_lat_demand / M_effective
            
            if a_lat_scaled >= a_grip_max:
                num = mu_eff + np.tan(theta_bank_effective)
                den = max(0.01, 1 - mu_eff * np.tan(theta_bank_effective))
                v_physics_limit = np.sqrt(g * R_here * (num / den))
                VLIMprofile[k] = min(VLIMprofile[k], v_physics_limit)
                v_next = VLIMprofile[k]
                a_grip_available = 0
            else:
                a_grip_available = np.sqrt(a_grip_max**2 - a_lat_scaled**2)
                
            a_brake_limit_local = min(a_mech_limit, a_grip_available)
            a_aero = (0.5 * rho * cd * frontalArea * v_next**2) / M_effective
            a_roll = (carMass * 9.81 * (Ad + Bd * v_next)) / M_effective
            a_grav = (carMass * 9.81 * sin_theta) / M_effective
            
            a_total_decel = max(0.01, (a_brake_limit_local * safety_factor) + a_aero + a_roll + a_grav)
            
            v_brake_limit = np.sqrt(v_next**2 + 2 * a_total_decel * dist)
            VLIMprofile[k] = min(VLIMprofile[k], v_brake_limit)
            
        v_start = VLIMprofile[0]
        v_end_max = np.sqrt(v_start**2 + 2 * (a_mech_limit * safety_factor) * segmentLengths[-1])
        VLIMprofile[-1] = min(VLIMprofile[-1], v_end_max)

    # =========================================================================
    # 3. Simulation loop
    # =========================================================================
    
    for i in range(numSteps):
        turnSign = TSignProfile[i]
        r_turn = RProfile[i]
        theta_bank = bankingProfile[i]
        ds_seg = segmentLengths[i]
        
        dz = zt[i+1] - zt[i]
        sin_theta = dz / np.sqrt(ds_seg**2 + dz**2)
        cos_theta = np.sqrt(1 - sin_theta**2)
        
        theta_bank_effective = np.sign(turnSign) * theta_bank
        
        if np.isfinite(r_turn) and r_turn > 0:
            phi_flat = np.arctan(velocity**2 / (g * r_turn))
            phi_i = phi_flat - theta_bank_effective
        else:
            phi_i = 0
            
        if (h_cog > t_tyre) and (phi_i != 0):
            phi_delta = np.arcsin((t_tyre * np.sin(phi_i)) / (h_cog - t_tyre))
        else:
            phi_delta = 0
            
        phi_mag = phi_i + phi_delta
        phi_signed_target = -np.sign(turnSign) * phi_mag
        
        phi_target = phi_signed_target
        if useMaxLeanClamp:
            phi_target = max(min(phi_target, maxLeanRad), -maxLeanRad)
            
        taper_start_angle = 0.7 * maxLeanRad
        if abs(phi_prev) > taper_start_angle:
            ratio_in_zone = (abs(phi_prev) - taper_start_angle) / (maxLeanRad - taper_start_angle)
            taper = 1 - min(ratio_in_zone, 1.0)**2
        else:
            taper = 1.0
            
        maxLeanRate_eff = maxLeanRate * taper
        if useLeanRateClamp:
            maxDeltaPhi = maxLeanRate_eff * ds_seg / max(velocity, 1.0)
            dphi_req = phi_target - phi_prev
            if abs(dphi_req) > maxDeltaPhi:
                phi_actual = phi_prev + maxDeltaPhi * np.sign(dphi_req)
            else:
                phi_actual = phi_target
        else:
            phi_actual = phi_target
            
        leanProfile[i] = phi_actual
        phi_prev = phi_actual
        phi_road_rel = abs(phi_actual)
        
        # Interp with fallback
        interp_func = interp1d(leanDeg_AreaTable, contactAreaTable, fill_value="extrapolate")
        A_contact = interp_func(np.rad2deg(phi_road_rel))
        Aprofile[i] = A_contact
        mu_adjust = (A_contact / A0)**0.15
        mu_eff = tireFrictionCoeff * mu_adjust
        Muprofile[i] = mu_eff
        
        if r_turn != np.inf:
            F_centrifugal_out = carMass * velocity**2 / r_turn * np.cos(theta_bank_effective)
            F_gravity_in = carMass * g * np.sin(theta_bank_effective)
            F_lat_demand = abs(F_centrifugal_out - F_gravity_in)
        else:
            F_lat_demand = 0
            
        K_v = K_vert[i]
        Fz_vert = carMass * (velocity**2) * K_v
        
        Fz = (carMass * g * np.cos(theta_bank_effective) * cos_theta) + \
             (carMass * (velocity**2 / max(1.0, r_turn)) * np.sin(theta_bank_effective)) + Fz_vert
        Fz = max(0.1, Fz)
        
        
        F_tire_total = mu_eff * Fz
        FTyreprofile[i] = F_tire_total
        
        if F_lat_demand > F_tire_total:
            F_long_cap = 0
        else:
            F_long_cap = np.sqrt(F_tire_total**2 - F_lat_demand**2)
            
        whl_rpm = (velocity * 30) / (np.pi * wheelRadius)
        
        if is_direct_drive:
            mot_rpm = whl_rpm * chainDriveRatio
            if mot_rpm <= motor_redline:
                T_avail = np.interp(mot_rpm, rpm_table, T_max_table)
            else:
                rpm_over = mot_rpm - motor_redline
                T_avail = max(0.0, T_max_table[-1] * (1 - (rpm_over / 500.0)))
                
            F_power_limit = (T_avail * chainDriveRatio * mech_eff) / wheelRadius
            active_eta = np.interp(mot_rpm, rpm_table, eta_max_table)
            if active_eta == 0: active_eta = 0.85 # Fallback
        else:
            if i > 0:
                if shift_timer > 0: shift_timer = max(0.0, shift_timer - DTprofile[i-1])
                if shift_cooldown_timer > 0: shift_cooldown_timer = max(0.0, shift_cooldown_timer - DTprofile[i-1])
                
            mot_rpm_current = whl_rpm * gear_ratios[current_gear] * finalDriveRatio
            best_gear = current_gear
            
            if shift_cooldown_timer <= 0:
                if current_gear < len(gear_ratios) - 1 and mot_rpm_current > 7400:
                    best_gear = current_gear + 1
                elif current_gear > 0:
                    mot_rpm_down = whl_rpm * gear_ratios[current_gear - 1] * finalDriveRatio
                    if mot_rpm_down < 7100:
                        best_gear = current_gear - 1
                        
            if best_gear != current_gear:
                shift_timer = shift_delay_time
                shift_cooldown_timer = shift_cooldown_time
                current_gear = best_gear
                
            mot_rpm_active = whl_rpm * gear_ratios[current_gear] * finalDriveRatio
            if mot_rpm_active <= motor_redline:
                T_avail = np.interp(mot_rpm_active, rpm_table, T_max_table)
            else:
                rpm_over = mot_rpm_active - motor_redline
                T_avail = max(0.0, T_max_table[-1] * (1 - (rpm_over / 500.0)))
                
            max_tractive_force = (T_avail * gear_ratios[current_gear] * finalDriveRatio * mech_eff) / wheelRadius
            active_eta = np.interp(mot_rpm_active, rpm_table, eta_max_table)
            if active_eta == 0: active_eta = 0.85
            
            F_power_limit = 0.0 if shift_timer > 0 else max_tractive_force

        v_target_next = VLIMprofile[i+1] if i < numSteps - 1 else VLIMprofile[i]
        
        a_req = (v_target_next**2 - velocity**2) / (2 * ds_seg)
        F_drag_est = 0.5 * cd * rho * frontalArea * velocity**2
        F_roll_est = carMass * g * (Ad + Bd * velocity)
        F_grav_est = carMass * g * sin_theta
        
        F_cmd = (a_req * M_effective) + F_drag_est + F_roll_est + F_grav_est
        
        if F_cmd >= 0:
            F_long = min(F_cmd, F_long_cap, F_power_limit)
        else:
            F_long = max(F_cmd, -F_brake_wheel_max)
            
        FCMDprofile[i] = F_cmd
        
        v_start = velocity
        F_drag = 0.5 * cd * rho * frontalArea * velocity**2
        F_roll = carMass * g * (Ad + Bd * velocity)
        F_grav = carMass * g * sin_theta
        
        V66 = min(speed_limit, maxV) * 0.95
        F_long_taper = F_long
        if (v_start > V66) and (F_long > 0):
            ratio = (v_start - V66) / (min(speed_limit, maxV) - V66)
            taper = max(0.0, 1.0 - ratio)
            F_long_taper = F_long * taper
            
        # Predictor-Corrector Integration (Heun's)
        F_net_start = F_long_taper - F_drag - F_roll - F_grav
        a_start = F_net_start / M_effective
        v_end_est = np.sqrt(max(0.1, v_start**2 + 2 * a_start * ds_seg))
        F_drag_end = 0.5 * cd * rho * frontalArea * v_end_est**2
        
        F_net_avg = F_long_taper - (F_drag + F_drag_end)/2 - F_roll - F_grav
        dv_avg = F_net_avg / M_effective
        v_sq = v_start**2 + 2 * dv_avg * ds_seg
        velocity = np.sqrt(max(0.1, v_sq))
        
        v_mean = (v_start + velocity) / 2
        dt_seg = ds_seg / max(0.1, v_mean)
        
        velocityProfile[i] = velocity
        timeProfile[i] = time_val
        accel[i] = dv_avg
        DTprofile[i] = dt_seg
        F_power_array[i] = F_power_limit
        F_remain_array[i] = F_long_cap
        F_applied[i] = F_long
        
        p_mech_wheel = F_long_taper * v_mean
        instPower = (p_mech_wheel / active_eta) + P_aux if p_mech_wheel > 0 else P_aux
        
        powerProfile[i] = instPower
        totalEnergy_J += (instPower * dt_seg)
        time_val += dt_seg

    # =========================================================================
    # 4. Final Outputs
    # =========================================================================
    telemetry = {
        'timeProfile': timeProfile,
        'velocityProfile': velocityProfile,
        'VLIMProfile': VLIMprofile,
        'accel': accel,
        'Fcom': np.column_stack((timeProfile, FCMDprofile)),
        'mu': np.column_stack((timeProfile, Muprofile)),
        'DC': np.column_stack((timeProfile, velocityProfile)),
        'leanProfile': leanProfile,
        'distance': s_track[:-1],
        'F_applied': F_applied
    }
    
    lap_time = timeProfile[-1]
    total_track_length = s_track[-1]
    avg_speed_ms = total_track_length / lap_time
    top_speed_kmh = np.max(velocityProfile) * 3.6
    
    print(f"Lap time:              {lap_time:.3f} s")
    print(f"Average vehicle speed: {avg_speed_ms:.2f} m/s ({avg_speed_ms * 3.6:.2f} km/h)")
    print(f"Top speed achieved:    {top_speed_kmh:.2f} km/h")

    # =========================================================================
    # 5. Plots & Export
    # =========================================================================
    if debug_mode:
        totalEnergy_Wh = totalEnergy_J / 3600
        totalEnergy_kWh = totalEnergy_Wh / 1000
        track_length_km = segmentLengths[-1] * numSteps / 1000
        print(f"Energy Consumed:       {totalEnergy_Wh:.2f} Wh ({totalEnergy_kWh:.4f} kWh)")
        
        # Example plotting (truncated for brevity, using Matplotlib)
        plt.figure('Velocity vs VLIM')
        plt.plot(timeProfile, velocityProfile, 'r', linewidth=1.5, label='Actual Speed')
        plt.plot(timeProfile, VLIMprofile, 'g', linewidth=1.5, label='Speed Limit (Grip/Braking)')
        plt.xlabel('Time (s)')
        plt.ylabel('Speed (m/s)')
        plt.title('Vehicle Speed vs Physical Limit')
        plt.legend()
        plt.grid(True)
        plt.show()
        
        # CSV Export using Pandas
        wheelbase = 1.4
        steerAngle_mag_rad = (wheelbase * np.abs(raw_curvature[:-1])) * np.cos(np.abs(leanProfile))
        steerAngle_deg = np.rad2deg(steerAngle_mag_rad) * np.sign(TSignProfile[:-1])
        
        # Handle zero division safely for inputs
        power_safe = np.where(F_power_array > 0.1, F_power_array, 0.1)
        accel_input = np.clip(np.maximum(0, F_applied) / power_safe, 0, 1)
        brake_input = np.clip(np.abs(np.minimum(0, F_applied)) / F_brake_wheel_max, 0, 1)
        
        telemetry_df = pd.DataFrame({
            'Distance_m': s_track[:-1],
            'Time_s': timeProfile,
            'Velocity_ms': velocityProfile,
            'VLIM_ms': VLIMprofile,
            'SteerAngle_deg': steerAngle_deg,
            'Throttle_0to1': accel_input,
            'Brake_0to1': brake_input,
            'Lean_deg': np.rad2deg(leanProfile)
        })
        
        csv_filename = 'Aragon_Telemetry.csv'
        telemetry_df.to_csv(csv_filename, index=False)
        print(f"Telemetry exported to {csv_filename}")

    return lap_time, totalEnergy_J, top_speed_kmh, telemetry

# To run it directly:
# if __name__ == "__main__":
#     bzero_v4_release()